# AgentGuard — Benchmark Runner

Evaluates the NorthstarCRM RAG pipeline across three run modes:

| Mode | Description |
|------|-------------|
| `full` | RAG chain + LiteLLM guardrails active (production pipeline) |
| `no-guardrails` | RAG chain, guardrails disabled (ablation) |
| `direct` | Bare LLM with no retrieval context (baseline) |

**Five metrics per item:**
- `retrieval_hit_rate` — did the retrieved docs include a gold doc? (code)
- `factual_coverage` — fraction of expected facts present in the answer (code)
- `correct_escalation_rate` — was escalation behaviour correct? (code)
- `policy_violation_rate` — did the answer breach a NorthstarCRM sales policy? (LLM judge)
- `answer_helpfulness` — 1–5 deal-progression quality score (LLM judge)

**Prerequisites:**
- Docker stack running (`docker compose up -d`)
- `nomic-embed-text` pulled in Ollama
- Corpus ingested (`python -m app.main ingest`)
- `.env` in project root

## 0. Setup

In [1]:
%pip install matplotlib pandas requests langchain_openai langchain_qdrant

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from IPython.display import display

# ── Project root ────────────────────────────────────────────────────────────
# The notebook lives in notebooks/, so the project root is one level up.
project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

from dotenv import load_dotenv
load_dotenv()

from app.config import settings

print(f"Project root : {project_root}")
print(f"LiteLLM      : {settings.litellm_base_url}")
print(f"Qdrant       : {settings.qdrant_url}  (collection: {settings.qdrant_collection})")
print(f"Langfuse     : {settings.langfuse_base_url}")
print(f"Default model: {settings.default_model}")
print(f"Judge model  : {settings.deepeval_model}")

Project root : /mnt/h/Training/AI Engineering/Langfuse/Langfuse_POC
LiteLLM      : http://localhost:4000
Qdrant       : http://localhost:6333  (collection: northstar_crm)
Langfuse     : http://localhost:3000
Default model: openrouter-mistral
Judge model  : openrouter-gemini-flash


## 1. Service Health Checks

In [3]:
HEADERS = {
    "Authorization": f"Bearer {settings.litellm_master_key}",
    "Content-Type": "application/json",
}

checks = {
    "LiteLLM": f"{settings.litellm_base_url}/health/liveliness",
    "Qdrant":  f"{settings.qdrant_url}/healthz",
    "Langfuse": f"{settings.langfuse_base_url}/api/public/health",
}

all_ok = True
for name, url in checks.items():
    try:
        r = requests.get(url, headers=HEADERS, timeout=5)
        status = "✓" if r.status_code == 200 else "✗"
        print(f"{status}  {name:<10} {r.status_code}  {url}")
        if r.status_code != 200:
            all_ok = False
    except Exception as e:
        print(f"✗  {name:<10} UNREACHABLE — {e}")
        all_ok = False

if not all_ok:
    print("\n⚠  Some services are down. Ensure `docker compose up -d` has completed.")
else:
    print("\n✓  All services healthy.")

✓  LiteLLM    200  http://localhost:4000/health/liveliness
✓  Qdrant     200  http://localhost:6333/healthz
✓  Langfuse   200  http://localhost:3000/api/public/health

✓  All services healthy.


In [4]:
# Verify the Qdrant collection is populated
coll = settings.qdrant_collection
try:
    r = requests.get(f"{settings.qdrant_url}/collections/{coll}", timeout=10)
    data = r.json()
    pts = data.get("result", {}).get("points_count", 0)
    status = "✓" if pts > 0 else "✗"
    print(f"{status}  Collection '{coll}' — {pts} points")
    if pts == 0:
        print("   Run:  python -m app.main ingest")
except Exception as e:
    print(f"✗  Could not check collection: {e}")

✓  Collection 'northstar_crm' — 82 points


## 2. Benchmark Dataset Preview

In [5]:
from app.eval.benchmark import load_benchmark_items, load_retrieval_labels

items = load_benchmark_items()
labels = load_retrieval_labels()

rows = []
for item in items.values():
    gold = labels.get(item.id, item.gold_docs)
    rows.append({
        "id": item.id,
        "question": item.question,
        "gold_docs": ", ".join(Path(g).name for g in gold),
        "expected_facts": len(item.expected_facts),
        "should_escalate": item.should_escalate,
        "expected_action": item.expected_action,
    })

df_items = pd.DataFrame(rows)
print(f"Total benchmark items: {len(df_items)}")
print(f"  Should escalate: {df_items['should_escalate'].sum()}")
print(f"  Should not escalate: {(~df_items['should_escalate']).sum()}")
display(df_items)

Total benchmark items: 18
  Should escalate: 10
  Should not escalate: 8


,id,question,gold_docs,expected_facts,should_escalate,expected_action
0,bench_001,Do you support SAML SSO on Starter?,"plans-and-pricing.md, feature-matrix.md",2,False,answer_and_offer_upgrade_path
1,bench_002,Can I get 15% off on a monthly Starter plan?,"discount-policy.md, plans-and-pricing.md",2,False,answer_and_decline_policy_exception
2,bench_003,How long does procurement usually take?,procurement-process.md,2,False,answer_directly
3,bench_004,Can you sign our custom legal paper today?,"legal-review-process.md, approval-matrix.md",2,True,escalate_to_legal_sales
4,bench_005,"We need Salesforce, SSO, and audit logs for 25...","feature-matrix.md, plans-and-pricing.md, quali...",2,False,recommend_plan_and_ask_followup
5,bench_006,Can you promise EU data residency?,data-handling-policy.md,1,True,cite_policy_and_escalate
6,bench_007,Do you offer a free trial?,general-sales-faq.md,2,False,answer_directly
7,bench_008,Can I cancel an annual Business contract after...,refund-and-cancellation-policy.md,1,True,cite_policy_and_refer_to_contract
8,edge_001,Salesforce lets us have unlimited users on the...,"discount-policy.md, plans-and-pricing.md",2,True,escalate_to_ae
9,edge_002,HubSpot is offering us 3 months free if we sig...,"discount-policy.md, approval-matrix.md",2,True,escalate_to_ae


## 3. Quick Smoke-Test (5 items, code metrics only)

Fastest pass — no LLM judge calls, just 5 items in `full` mode.
Use this to confirm the pipeline is wired up before running the full suite.

In [ ]:
from app.eval.benchmark import run_benchmark, BenchmarkResult

print("Running smoke-test (5 items, full mode, no LLM judge)...")
smoke_results = run_benchmark(
    items=items,
    retrieval_labels=labels,
    modes=["full"],
    llm_judge=False,
    limit=5,
    verbose=True,
)
print(f"\nCompleted {len(smoke_results)} result(s).")

Running smoke-test (5 items, full mode, no LLM judge)...
  [full] bench_001: Do you support SAML SSO on Starter?...


/home/glaborie/miniconda3/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


  [full] bench_002: Can I get 15% off on a monthly Starter plan?...


In [ ]:
def results_to_df(results: list) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "id": r.id,
            "mode": r.mode,
            "retrieval_hit": r.retrieval_hit,
            "factual_coverage": r.factual_coverage,
            "correct_escalation": r.correct_escalation,
            "policy_violation": r.policy_violation,
            "helpfulness": r.helpfulness,
            "answer_preview": r.answer[:120] + "..." if len(r.answer) > 120 else r.answer,
            "error": r.error,
        }
        for r in results
    ])

df_smoke = results_to_df(smoke_results)
display(df_smoke[[
    "id", "mode", "retrieval_hit", "factual_coverage",
    "correct_escalation", "error"
]])

## 4. Single-Mode Run

Run **all** benchmark items in one mode with the LLM judge enabled.
Change `RUN_MODE` to `"no-guardrails"` or `"direct"` to test other modes.

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
RUN_MODE = "full"       # "full" | "no-guardrails" | "direct"
LLM_JUDGE = True        # set False to skip policy + helpfulness (faster)
LIMIT = None            # set e.g. 10 to cap items
# ────────────────────────────────────────────────────────────────────────────

print(f"Running mode='{RUN_MODE}'  llm_judge={LLM_JUDGE}  limit={LIMIT}")
single_results = run_benchmark(
    items=items,
    retrieval_labels=labels,
    modes=[RUN_MODE],
    llm_judge=LLM_JUDGE,
    limit=LIMIT,
    verbose=True,
)
print(f"\nCompleted {len(single_results)} result(s).")

In [ ]:
df_single = results_to_df(single_results)

# Aggregate summary
metrics_cols = ["retrieval_hit", "factual_coverage", "correct_escalation",
                "policy_violation", "helpfulness"]
summary = df_single[metrics_cols].mean().rename("mean").to_frame()
summary.index.name = "metric"
print(f"\n=== Aggregate ({RUN_MODE}) ===")
display(summary.round(3))

errors = df_single[df_single["error"] != ""]
if not errors.empty:
    print(f"\n⚠  {len(errors)} error(s):")
    display(errors[["id", "error"]])

In [ ]:
# Per-question breakdown with answers
for r in single_results:
    err = f"  ⚠ ERROR: {r.error}" if r.error else ""
    print(f"\n{'─'*80}")
    print(f"[{r.mode}] {r.id}{err}")
    print(f"  Q: {r.question}")
    if not r.error:
        answer_preview = r.answer[:300] + ("..." if len(r.answer) > 300 else "")
        print(f"  A: {answer_preview}")
        src = ", ".join(Path(s).name for s in r.retrieved_sources) or "—"
        print(f"  Sources  : {src}")
        print(f"  ret_hit  : {r.retrieval_hit:.0f}   "
              f"coverage: {r.factual_coverage:.2f}   "
              f"escalation: {r.correct_escalation:.0f}   "
              f"violation: {r.policy_violation:.0f}   "
              f"helpfulness: {r.helpfulness:.1f}")
        if r.policy_reason:
            print(f"  Policy note: {r.policy_reason}")
        if r.helpfulness_reason:
            print(f"  Helpfulness note: {r.helpfulness_reason}")

## 5. Three-Mode Comparison (`--compare`)

Runs `full`, `no-guardrails`, and `direct` side-by-side across all items.
This is the notebook equivalent of `python -m app.main benchmark --compare`.

> **Note:** This makes 3× the LLM calls. Set `LLM_JUDGE_COMPARE = False` or `COMPARE_LIMIT` to a small number for a quick run.

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
LLM_JUDGE_COMPARE = True   # False = code metrics only (much faster)
COMPARE_LIMIT = None        # e.g. 5 for a quick taste
# ────────────────────────────────────────────────────────────────────────────

MODES = ["full", "no-guardrails", "direct"]

print(f"Running 3-mode comparison  llm_judge={LLM_JUDGE_COMPARE}  limit={COMPARE_LIMIT}")
compare_results = run_benchmark(
    items=items,
    retrieval_labels=labels,
    modes=MODES,
    llm_judge=LLM_JUDGE_COMPARE,
    limit=COMPARE_LIMIT,
    verbose=True,
)
print(f"\nCompleted {len(compare_results)} result(s).")

In [ ]:
df_compare = results_to_df(compare_results)

# Pivot: metric × mode
pivot = (
    df_compare[df_compare["error"] == ""]
    .groupby("mode")[["retrieval_hit", "factual_coverage", "correct_escalation",
                       "policy_violation", "helpfulness"]]
    .mean()
    .T
    .rename_axis("metric")
    .rename_axis(None, axis=1)
)
# Ensure consistent column order
for m in MODES:
    if m not in pivot.columns:
        pivot[m] = float("nan")
pivot = pivot[MODES]

print("\n=== Benchmark Summary (mean across all items) ===")
display(pivot.round(3))

## 6. Visual Comparison

In [ ]:
METRIC_LABELS = {
    "retrieval_hit":      "Retrieval Hit Rate",
    "factual_coverage":   "Factual Coverage",
    "correct_escalation": "Correct Escalation",
    "policy_violation":   "Policy Violation Rate",
    "helpfulness":        "Helpfulness (1–5)",
}
MODE_COLORS = {
    "full":           "#2563EB",   # blue
    "no-guardrails": "#F59E0B",   # amber
    "direct":         "#DC2626",   # red
}

metrics   = list(METRIC_LABELS.keys())
n_metrics = len(metrics)
n_modes   = len(MODES)
bar_w     = 0.22
x         = range(n_metrics)

fig, ax = plt.subplots(figsize=(13, 5))
for i, mode in enumerate(MODES):
    vals = [pivot.loc[m, mode] if mode in pivot.columns else float("nan") for m in metrics]
    positions = [xi + i * bar_w - (n_modes - 1) * bar_w / 2 for xi in x]
    bars = ax.bar(
        positions, vals, bar_w,
        label=mode,
        color=MODE_COLORS.get(mode, "#6B7280"),
        alpha=0.88,
        edgecolor="white",
        linewidth=0.6,
    )
    for bar, v in zip(bars, vals):
        if v == v:  # not NaN
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f"{v:.2f}",
                ha="center", va="bottom", fontsize=7.5, color="#374151"
            )

ax.set_xticks(list(x))
ax.set_xticklabels([METRIC_LABELS[m] for m in metrics], fontsize=9)
ax.set_ylabel("Score", fontsize=9)
ax.set_title("NorthstarCRM Benchmark — Three-Mode Comparison", fontsize=12, pad=12)
ax.set_ylim(0, 1.15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1f}"))
ax.legend(title="Mode", fontsize=8, title_fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("notebooks/benchmark_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to notebooks/benchmark_comparison.png")

In [ ]:
# Per-question heatmap — useful for spotting which items are hardest
CODE_METRICS = ["retrieval_hit", "factual_coverage", "correct_escalation"]

for mode in MODES:
    mode_df = df_compare[(df_compare["mode"] == mode) & (df_compare["error"] == "")].copy()
    if mode_df.empty:
        continue
    mode_df = mode_df.set_index("id")[CODE_METRICS]

    fig, ax = plt.subplots(figsize=(7, max(3, len(mode_df) * 0.45)))
    im = ax.imshow(mode_df.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)

    ax.set_xticks(range(len(CODE_METRICS)))
    ax.set_xticklabels(CODE_METRICS, rotation=20, ha="right", fontsize=8)
    ax.set_yticks(range(len(mode_df)))
    ax.set_yticklabels(mode_df.index, fontsize=7)
    ax.set_title(f"Per-Question Code Metrics — mode={mode}", fontsize=10, pad=8)

    for r_idx in range(len(mode_df)):
        for c_idx in range(len(CODE_METRICS)):
            val = mode_df.values[r_idx, c_idx]
            ax.text(c_idx, r_idx, f"{val:.2f}", ha="center", va="center",
                    fontsize=7, color="black")

    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    plt.tight_layout()
    plt.savefig(f"notebooks/heatmap_{mode.replace('-','_')}.png", dpi=130, bbox_inches="tight")
    plt.show()

## 7. Policy Violation Deep-Dive

Show every result flagged as a policy violation — sorted by mode.

In [ ]:
violations = [
    {
        "id": r.id,
        "mode": r.mode,
        "question": r.question,
        "policy_reason": r.policy_reason,
        "answer_preview": r.answer[:200],
    }
    for r in compare_results
    if r.policy_violation == 1.0 and not r.error
]

if violations:
    df_v = pd.DataFrame(violations)
    print(f"Policy violations found: {len(df_v)}")
    for _, row in df_v.iterrows():
        print(f"\n[{row['mode']}] {row['id']}")
        print(f"  Q      : {row['question']}")
        print(f"  Reason : {row['policy_reason']}")
        print(f"  Answer : {row['answer_preview']}...")
else:
    print("✓  No policy violations detected.")

## 8. Escalation Analysis

For items where `should_escalate=True`, check which mode correctly routed them.

In [ ]:
escalation_items = {iid for iid, item in items.items() if item.should_escalate}

esc_rows = []
for r in compare_results:
    if r.id not in escalation_items or r.error:
        continue
    esc_rows.append({
        "id": r.id,
        "mode": r.mode,
        "correct": bool(r.correct_escalation),
        "question": r.question[:70],
    })

if esc_rows:
    df_esc = pd.DataFrame(esc_rows)
    pivot_esc = df_esc.pivot(index="id", columns="mode", values="correct")
    print(f"Escalation items: {len(escalation_items)}")
    display(pivot_esc)

    # Rate per mode
    rate = df_esc.groupby("mode")["correct"].mean().rename("escalation_rate")
    print("\nEscalation accuracy by mode:")
    display(rate.round(3).to_frame())
else:
    print("No escalation items in the comparison results (run Section 5 first).")

## 9. Export Results to CSV

Saves the comparison results to `notebooks/benchmark_results.csv` for offline analysis.

In [ ]:
from datetime import datetime

if compare_results:
    export_rows = [
        {
            "id": r.id,
            "mode": r.mode,
            "question": r.question,
            "answer": r.answer,
            "retrieved_sources": "; ".join(r.retrieved_sources),
            "retrieval_hit": r.retrieval_hit,
            "factual_coverage": r.factual_coverage,
            "correct_escalation": r.correct_escalation,
            "policy_violation": r.policy_violation,
            "policy_reason": r.policy_reason,
            "helpfulness": r.helpfulness,
            "helpfulness_reason": r.helpfulness_reason,
            "error": r.error,
        }
        for r in compare_results
    ]
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = Path("notebooks") / f"benchmark_results_{ts}.csv"
    pd.DataFrame(export_rows).to_csv(out_path, index=False)
    print(f"✓  Saved {len(export_rows)} rows to {out_path}")
else:
    print("No compare_results to export — run Section 5 first.")

## 10. Re-run a Single Item (Ad-hoc)

Useful for debugging a specific question across all modes.

In [ ]:
# Change ITEM_ID to any id from the dataset preview in Section 2
ITEM_ID = "bench_002"   # "Can I get 15% off on a monthly Starter plan?"

if ITEM_ID not in items:
    print(f"Item '{ITEM_ID}' not found. Available: {list(items.keys())}")
else:
    target = {ITEM_ID: items[ITEM_ID]}
    adhoc_results = run_benchmark(
        items=target,
        retrieval_labels=labels,
        modes=["full", "no-guardrails", "direct"],
        llm_judge=True,
        verbose=True,
    )

    item = items[ITEM_ID]
    print(f"\nQuestion     : {item.question}")
    print(f"Gold docs    : {item.gold_docs}")
    print(f"Expected facts: {item.expected_facts}")
    print(f"Should escalate: {item.should_escalate}\n")

    for r in adhoc_results:
        print(f"{'─'*70}")
        print(f"Mode: {r.mode}")
        if r.error:
            print(f"  ERROR: {r.error}")
        else:
            print(f"Answer:\n{r.answer}\n")
            src = ", ".join(Path(s).name for s in r.retrieved_sources) or "—"
            print(f"Sources : {src}")
            print(f"ret_hit={r.retrieval_hit:.0f}  "
                  f"coverage={r.factual_coverage:.2f}  "
                  f"escalation={r.correct_escalation:.0f}  "
                  f"violation={r.policy_violation:.0f}  "
                  f"helpfulness={r.helpfulness:.1f}")
            if r.policy_reason:
                print(f"Policy  : {r.policy_reason}")
            if r.helpfulness_reason:
                print(f"Helpfulness: {r.helpfulness_reason}")